# DominusUltra independent GPU verification

This notebook clones the public repository, records the exact commit and GPU/software environment, runs correctness-gated prefill and decode cases, and produces raw JSON plus a readable Markdown report. A benchmark number is valid only when the final verdict is **PASS**.

In [ ]:
import os
import subprocess
import sys

repo_url = "https://github.com/MiMindMendinc/DominusUltra.git"
target_ref = "agent/reproducible-gpu-evidence"
repo_dir = "/content/DominusUltra"
if not os.path.isdir(os.path.join(repo_dir, ".git")):
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
os.chdir(repo_dir)
subprocess.run(["git", "fetch", "origin", target_ref], check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
assert not status, f"Expected a clean checkout, found:\n{status}"
print("Target ref:", target_ref)
print("Commit:", commit)

In [ ]:
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
subprocess.run([
    sys.executable,
    "gpu_evidence.py",
    "--suite", "quick",
    "--dtype", "auto",
    "--warmup", "10",
    "--iterations", "50",
], check=True)

## Download and submit

The next cell downloads a ZIP containing the raw JSON and Markdown report. Attach both files to a new [Benchmark result issue](https://github.com/MiMindMendinc/DominusUltra/issues/new?template=benchmark_result.md). Failed runs are useful too—please include them unchanged.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/dominus-ultra-evidence", "zip", "benchmark_results")
files.download(archive)